<a href="https://colab.research.google.com/github/hamsamahmoud-cpu/DistilBERT-Fine-tuning-for-Movie-Review-Sentiment-Analysis1/blob/main/02_distilbert_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries (in case your Colab session restarted)
!pip install -U transformers datasets evaluate accelerate

import os
import numpy as np
import evaluate
from google.colab import drive
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)

# Mount Drive
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/distilbert-imdb-sentiment'
DATA_PATH = os.path.join(PROJECT_PATH, "tokenized_data")
MODEL_SAVE_PATH = os.path.join(PROJECT_PATH, "distilbert-finetuned")

# Load the preprocessed data from Week 1
print("Loading tokenized data from Drive...")
tokenized_datasets = load_from_disk(DATA_PATH)
print("Data loaded successfully!")

In [ ]:
# Shuffle and select a small subset for testing the pipeline
train_subset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
eval_subset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))
print(f"Training on {len(train_subset)} samples, evaluating on {len(eval_subset)} samples.")

In [ ]:
print("Loading DistilBERT model...")
model_ckpt = "distilbert-base-uncased"
# num_labels=2 because reviews are either Positive (1) or Negative (0)
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)

# FREEZE the bottom 4 layers
for layer in model.distilbert.transformer.layer[:4]:
    for param in layer.parameters():
        param.requires_grad = False

print("Successfully frozen the bottom 4 layers. Only the top 2 and the head will train.")

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert raw model outputs (logits) into 0 or 1 predictions
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    learning_rate=2e-5,
    per_device_train_batch_size=4,       # Keeps RAM usage strictly low
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,       # The CPU trick: Effective batch size = 16
    num_train_epochs=1,                  # Keep it at 1 for this test run
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"                     # Prevents logging clutter
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    compute_metrics=compute_metrics,
)

In [ ]:
print("Starting fine-tuning process...")
trainer.train()

# Save the finalized model and tokenizer
trainer.save_model(MODEL_SAVE_PATH)
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print(f"Week 2 Complete! Fine-tuned model saved to {MODEL_SAVE_PATH}")